# Assignment 1: Vector Database Creation and Retrieval
## Day 7 - RAG Fundamentals

The goal is not to build a completely new system from scratch. The goal is to repeat the same flow with guided TODOs so the first notebook feels intuitive.

## What you will build

You will build a small multimodal RAG retrieval pipeline using LlamaIndex and LanceDB:

1. Mount Google Drive and install dependencies
2. Configure LlamaIndex settings
3. Explore a folder containing different file types
4. Load documents using `SimpleDirectoryReader`
5. Create a LanceDB vector store
6. Create a `StorageContext`
7. Build a `VectorStoreIndex`
8. Retrieve relevant chunks for a query
9. Inspect retrieved chunks and metadata
10. Generate an answer by sending the retrieved chunks to an LLM through OpenRouter

## Main idea

Yesterday's session and notebooks showed the full RAG system end-to-end. This assignment asks you to complete the similar flow step by step.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it

## 0. Mount Google Drive

Use this if you are running the notebook in Google Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install dependencies

Use the same requirements file path used in class. If your folder path is different, update it below.

In [2]:
!pip install -q -r "/content/drive/MyDrive/outskill_c11/rag/requirements.txt" # this path will change for you
!pip install -q gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!apt-get update && apt-get install -y ffmpeg
!pip install openai-whisper

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,371 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.

## 2. Set OpenRouter API key

We use OpenRouter for the LLM, similar to the class notebook. The embedding model will be local.

In [33]:
import os
from google.colab import userdata

# securely input your key
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
print("✓ OpenRouter key set successfully")

✓ OpenRouter key set successfully


## 3. Imports and configuration

Before running the full pipeline, check these values:

- `data_path`: folder that contains your input files (CHANGE IT ACCORDING TO YOUR OWN PATH)
- `vector_db_path`: where LanceDB will store vectors
- `index_storage_path`: where LlamaIndex will persist index information
- `chunk_size` and `chunk_overlap`: same concepts discussed in class

In [5]:
# Import required libraries
import os
from pathlib import Path
import time
from typing import List

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
CONFIG = {
    "llm_model": "openai/gpt-5-mini",   # OpenRouter model slug
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,
    "data_path": "/content/drive/MyDrive/outskill_c11/rag/data",  # change the path accordingly
    "vector_db_path": "/content/storage/assignment_multimodal_vectordb",
    "index_storage_path": "/content/storage/assignment_multimodal_index",
    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: /content/drive/MyDrive/outskill_c11/rag/data
Vector DB path: /content/storage/assignment_multimodal_vectordb
Index storage path: /content/storage/assignment_multimodal_index


## 4. Configure LlamaIndex settings

This step defines:

- which embedding model will convert text into vectors
- how text will be chunked

The LLM is not configured here. In this assignment LlamaIndex only handles loading, chunking, embedding and retrieval. We call the LLM directly through OpenRouter in section 11.

In [7]:
from llama_index.core import Settings
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.node_parser import SentenceSplitter

def configure_llamaindex_settings():
    """Configure LlamaIndex global settings using hardcoded configuration."""

    # Set up local embedding model (downloads locally first time, then cached)
    Settings.embed_model = resolve_embed_model(CONFIG["embedding_model"])
    print(f"✓ Embedding model configured: {CONFIG['embedding_model']}")

    # Set up node parser for chunking with hardcoded settings
    Settings.node_parser = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"]
    )
    print(f"✓ Text chunking configured: {CONFIG['chunk_size']} chars with {CONFIG['chunk_overlap']} overlap")

# Configure the settings
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured for multimodal processing")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap
✓ LlamaIndex settings configured for multimodal processing


## 5. Explore the dataset

Before loading documents, always inspect what files are present.

This helps you answer questions like:

- How many files are present?
- What file types are present?
- Are we working with PDFs only or multiple formats?
- Is the folder path correct?

In [8]:
def explore_dataset(data_path: str = None):
    """
    Explore and categorize the files in our dataset by type.

    Args:
        data_path (str): Path to the data directory
    """
    if data_path is None:
        data_path = CONFIG["data_path"]

    data_dir = Path(data_path)
    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return

    # Categorize files by type
    file_types = {}
    all_files = []

    # Walk through all files recursively
    for file_path in data_dir.rglob("*"):
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            file_size = file_path.stat().st_size

            if suffix not in file_types:
                file_types[suffix] = []

            file_info = {
                "path": str(file_path),
                "name": file_path.name,
                "size_mb": round(file_size / (1024 * 1024), 2),
                "size_bytes": file_size
            }

            file_types[suffix].append(file_info)
            all_files.append(file_info)

    # Display summary
    print("---Dataset Overview---")
    print(f"Total files found: {len(all_files)}")

    print(f"\nFile Types Distribution:")
    for file_type, files in sorted(file_types.items()):
        if file_type:  # Skip files without extension
            total_size = sum(f["size_mb"] for f in files)
            print(f"  {file_type}: {len(files)} files ({total_size:.2f} MB)")

            # Show file details
            for file_info in files[:3]:  # Show first 3 files of each type
                print(f"    - {file_info['name']} ({file_info['size_mb']} MB)")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")

            print()

    return file_types, all_files

# Explore our dataset
file_types, all_files = explore_dataset()
print(f"✓ Found {len(all_files)} files across {len(file_types)} different file types")


---Dataset Overview---
Total files found: 21

File Types Distribution:
  .csv: 4 files (0.00 MB)
    - agent_evaluation_metrics.csv (0.0 MB)
    - agent_performance_benchmark.csv (0.0 MB)
    - investment_portfolio.csv (0.0 MB)
    ... and 1 more

  .html: 2 files (0.00 MB)
    - agent_tutorial.html (0.0 MB)
    - fitness_tracker.html (0.0 MB)

  .md: 4 files (0.00 MB)
    - city_guides.md (0.0 MB)
    - agent_framework_comparison.md (0.0 MB)
    - market_analysis.md (0.0 MB)
    ... and 1 more

  .mp3: 3 files (2.95 MB)
    - ai_agents.mp3 (1.54 MB)
    - in_the_end.mp3 (0.6 MB)
    - rags.mp3 (0.81 MB)

  .pdf: 2 files (1.92 MB)
    - AI_Agent_Frameworks.pdf (0.34 MB)
    - Emerging_Agent_Architectures.pdf (1.58 MB)

  .png: 6 files (0.55 MB)
    - agent_performance_comparison.png (0.17 MB)
    - agent_types_comparison.png (0.1 MB)
    - fitness_progress.png (0.08 MB)
    ... and 3 more

✓ Found 21 files across 6 different file types


## 6. Load multimodal documents

In the class notebook, we used `SimpleDirectoryReader` because it can load many file types such as PDF, CSV, Markdown, HTML, images, and notebooks.

At this stage, files become LlamaIndex `Document` objects.

### Your task

Complete the function below to load documents from the dataset folder.

In [9]:
from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(data_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    TODO: Complete this function to load documents from the given folder path.
    HINT: Use SimpleDirectoryReader with recursive parameter to load all files

    Args:
        data_path (str): Path to the folder containing data

    Returns:
        List of documents loaded from the folder
    """

    if data_path is None:
        data_path = CONFIG["data_path"]


    # TODO: Create SimpleDirectoryReader (HINT: use recursive=True)
    reader = SimpleDirectoryReader(
        input_dir=data_path,
        recursive=True
    )

    # TODO: Load and return documents
    documents = reader.load_data()

    return documents

# Test the function after you complete it
test_folder = CONFIG["data_path"]
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 148MiB/s]
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Loaded 42 documents


## 7. Create LanceDB vector store

The vector store is where embeddings are stored and searched.

Important idea:

- LlamaIndex creates chunks and embeddings
- LanceDB stores the vector representation
- Later, the retriever searches this vector store

### Your task

Complete the function below.

In [10]:
import lancedb
# Vector store and index creation
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

def create_vector_store(vector_db_path: str):
    """
    Create a LanceDB vector store for storing document embeddings.

    TODO: In this function, you need to:
    1. Create the database directory if it does not already exist.
    2. Create a LanceDBVectorStore object.
    3. Return the vector store.

    Args:
        db_path (str): Path where the vector database will be stored.

    Returns:
        LanceDBVectorStore: Configured vector store.
    """

    # TODO: Create the directory if it doesn't exist
    Path(vector_db_path).mkdir(parents=True, exist_ok=True)

    # Connect to LanceDB (creates a connection to the LanceDB database)
    db = lancedb.connect(str(vector_db_path))
    print(f"✓ Connected to LanceDB at: {vector_db_path}")

    # TODO: Create vector store (HINT: Use LanceDBVectorStore with uri=str(vector_db_path) and table_name=CONFIG["table_name"])
    vector_store = LanceDBVectorStore(
        uri=str(vector_db_path),
        table_name="multimodal_documents"
    )

    print("✓ LanceDB vector store created for multimodal data")

    return vector_store


# Test the function after you complete it
vector_store = create_vector_store(vector_db_path=CONFIG["vector_db_path"])
print(f"Vector store created: {vector_store is not None}")

✓ Connected to LanceDB at: /content/storage/assignment_multimodal_vectordb
✓ LanceDB vector store created for multimodal data
Vector store created: True


## 8. Create StorageContext and VectorStoreIndex

This is one of the most important parts of the assignment.

### What is `StorageContext`?

Think of it as the object that tells LlamaIndex where the prepared index data should live.

In this assignment:

- `vector_store` stores embeddings
- `StorageContext` connects LlamaIndex to that vector store
- `VectorStoreIndex.from_documents()` builds the index from documents

### Your task

Complete the function below.

In [22]:
def create_vector_index(documents: List,
                        vector_store,
                        index_storage_path: str = None,
                        force_rebuild: bool = False):
    """
    Create a VectorStoreIndex from loaded documents.

    Returns:
        VectorStoreIndex object.
    """
    if index_storage_path is None:
        index_storage_path = CONFIG["index_storage_path"]

    index_path = Path(index_storage_path)
    index_path.mkdir(parents=True, exist_ok=True)

    # Check if index already exists
    # index_store_file = index_path / "index_store.json"

    print("Creating storage context")
    # TODO: Create storage context from vector_store (HINT: Use StorageContext)
    # storage_context = ?
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store
    )

    # TODO: Create the VectorStoreIndex from documents
    # index = ?
    index = VectorStoreIndex.from_documents(
        documents=documents,
        storage_context=storage_context
    )
    print("✓ Successfully loaded existing multimodal index")

    # TODO: Persist index metadata (uncomment the below lines)
    index.storage_context.persist(persist_dir=str(index_path))

    return index

if vector_store and documents:
    index = create_vector_index(documents, vector_store)

Creating storage context
✓ Successfully loaded existing multimodal index


## 9. Create a retriever
A retriever does not generate an answer. It only returns the most relevant chunks.

This is useful because it lets you inspect what the RAG system found before the LLM answers.

### Your task

Complete the retriever setup.

In [23]:
from llama_index.core.retrievers import VectorIndexRetriever

def create_retriever(index, similarity_top_k: int = None):
    """
    Create a retriever from the index.

    Args:
        index: VectorStoreIndex object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        VectorIndexRetriever object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]

    # TODO: Create VectorIndexRetriever
    # retriever =
    retriever = VectorIndexRetriever(
          index=index,
          similarity_top_k=similarity_top_k,
    )

    print(f"Retriever created with similarity_top_k={similarity_top_k}")
    return retriever

retriever = create_retriever(index)

Retriever created with similarity_top_k=5


## 10. Retrieve chunks for a query

This step helps you see exactly what semantic search returns.

Good queries to try:

- Ask about a topic you know exists in the dataset
- Ask using different wording than the original document
- Ask a broad question and inspect whether results are noisy

In [25]:
def retrieve_chunks(retriever, query: str, show_text: bool = True):
    """
    Retrieve relevant chunks for a query and print their metadata and score.

    Args:
        retriever: VectorIndexRetriever object.
        query: User query.
        show_text: Whether to print text previews.

    Returns:
        Retrieved nodes.
    """
    print("Query:", query)
    print("=" * 80)

    nodes = retriever.retrieve(query)

    print("Retrieved chunks:", len(nodes))

    for i, node_with_score in enumerate(nodes, 1):
        node = node_with_score.node
        score = node_with_score.score
        metadata = node.metadata

        print(f"Result {i}")
        print("Score:", score)
        print("File name:", metadata.get("file_name", "unknown"))
        print("File type:", metadata.get("file_type", "unknown"))

        if show_text:
            print("Text preview:")
            print(node.get_content()[:700])

        print("-" * 80)

    return nodes

sample_query = "what are the Best Practices for Cloud Python Environments"
retrieved_nodes = retrieve_chunks(retriever, sample_query)

Query: what are the Best Practices for Cloud Python Environments
Retrieved chunks: 5
Result 1
Score: 0.6118687391281128
File name: AI_Agent_Frameworks.pdf
File type: application/pdf
Text preview:
• Cost Efficiency : [3] reports cloud-based agents can re-
duce infrastructure costs by 30-40% compared to on-
premise solutions for equivalent workloads.
The evolution of these cloud Python libraries has lowered
the barrier to entry for financial institutions adopting agentic
AI, while providing the security and compliance features
required in regulated environments [68].
E. Agno: Cloud-Native Agent Framework
Agno is a Python framework designed for building and
deploying LLM-powered agents in the cloud, with features for
multi-agent orchestration, cloud deployment, and integration
with major providers such as AWS [30]. Agno supports both
local and cloud workflows, offering a built-in agent 
--------------------------------------------------------------------------------
Result 2
Score: 0.5975

## 11. Generate an answer with OpenRouter

A retriever only returns chunks. To get a final answer we do two more steps ourselves:

- **Augmentation**: put the retrieved chunks and the user's question into one prompt
- **Generation**: send that prompt to an LLM and read back the answer

We call the LLM directly through OpenRouter. OpenRouter is OpenAI-compatible, so we use the `openai` library and only change the `base_url`.

### Your task

Complete the function below.

In [34]:
from openai import OpenAI

# OpenRouter client (same openai library, different base_url)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

SYSTEM_PROMPT = (
    "Answer the question using only the context below. Each context block starts with its "
    "source file name. If the context does not contain the answer, say so."
) # TRY TO CREATE AND IMPROVE YOUR OWN PROMPT HERE

def build_context(nodes) -> str:
    """Join the retrieved chunks into one context string, each labelled with its source file."""
    return "\n\n".join(
        f"[Source: {n.metadata.get('file_name', 'unknown')}]\n{n.get_content()}"
        for n in nodes
        if n.get_content().strip()   # skip empty chunks (for example images with no text)
    )

def generate_answer(question: str, nodes) -> str:
    """
    Generate an answer from retrieved chunks by calling OpenRouter directly.

    Args:
        question: User question.
        nodes: Chunks returned by the retriever.

    Returns:
        Answer text from the LLM.
    """
    context = build_context(nodes)

    # TODO: Build the messages list (this is the augmentation step)
    # HINT: a list with two dicts:
    #   {"role": "system", "content": SYSTEM_PROMPT}
    #   {"role": "user", "content": <the context followed by the question>}
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{context}\n\nQuestion: {question}"}
    ]

    # TODO: Call OpenRouter (HINT: client.chat.completions.create with model=CONFIG["llm_model"] and messages=messages)
    response = client.chat.completions.create(
        model=CONFIG["llm_model"],
        messages=messages
    )

    return response.choices[0].message.content

# Test the function after you complete it (uses the chunks retrieved in section 10)
print(generate_answer(sample_query, retrieved_nodes))

From the provided context, recommended best practices for cloud Python environments when deploying agentic systems are:

- Use isolated Python environments (e.g., venv) to manage dependencies and ensure reproducibility.  
- Leverage templates and pre‑configured codebases (e.g., Agno) to speed transition from prototype to production.  
- Build in monitoring and debugging support for distributed/cloud deployments.  
- Keep agents focused and iterate (start simple rather than trying to do everything at once).  
- Use clear, specific prompts/questions when interacting with agents.  
- Chunk large documents appropriately for processing.  
- Choose the right embedding/model for your domain.  
- Monitor and evaluate agent performance regularly.


## 12. Ask a question and inspect sources

This is the full RAG flow in one function: retrieve the chunks, generate the answer, then inspect which source chunks were used.

In [ ]:
def ask_question(retriever, question: str, show_sources: bool = True):
    """
    Retrieve chunks for a question, generate an answer, and display answer plus sources.

    Args:
        retriever: VectorIndexRetriever object.
        question: User question.
        show_sources: Whether to show source chunks.
    """
    print("Question:", question)
    print("=" * 80)

    nodes = retriever.retrieve(question)          # Retrieval (LlamaIndex)
    answer = generate_answer(question, nodes)     # Augmentation + Generation (OpenRouter)

    print("Answer:")
    print(answer)

    if show_sources:
        print("\nSources used:")
        for i, source in enumerate(nodes, 1):
            metadata = source.node.metadata
            print(f"Source {i}")
            print("Score:", source.score)
            print("File name:", metadata.get("file_name", "unknown"))
            print("File type:", metadata.get("file_type", "unknown"))
            print("Text preview:")
            print(source.node.get_content()[:500].replace("\n", " "))
            print("-" * 80)

# Try your own question here
question = "What is the best time to visit Tokyo?"
ask_question(retriever, question, show_sources=True)

## 13. Launch the RAG app with Gradio

Now that the complete RAG pipeline is ready, use the interface below to ask questions without calling Python functions manually.

The app displays both the generated answer and the retrieved source chunks so that you can verify where the answer came from. Run this cell only after `retriever` and `generate_answer` are working.

When running in Google Colab, `share=True` creates a temporary public link. Do not enter private or sensitive information into a publicly shared app.

In [35]:
import gradio as gr

def answer_with_rag(question: str):
    """Return a RAG answer and the source chunks used to produce it."""
    if not question or not question.strip():
        return "Please enter a question.", "No sources to display."

    nodes = retriever.retrieve(question.strip())
    answer = generate_answer(question.strip(), nodes)
    source_sections = []

    for i, source in enumerate(nodes, 1):
        node = source.node
        metadata = node.metadata or {}
        score = "unknown" if source.score is None else f"{source.score:.4f}"
        preview = node.get_content()[:500].replace("\n", " ")

        source_sections.append(
            f"Source {i}\n"
            f"File: {metadata.get('file_name', 'unknown')}\n"
            f"Type: {metadata.get('file_type', 'unknown')}\n"
            f"Score: {score}\n"
            f"Preview: {preview}"
        )

    sources = "\n\n".join(source_sections) or "No source nodes were returned."
    return answer, sources

demo = gr.Interface(
    fn=answer_with_rag,
    inputs=gr.Textbox(
        label="Question",
        placeholder="Ask a question about your documents",
        lines=2,
    ),
    outputs=[
        gr.Textbox(label="Answer", lines=8),
        gr.Textbox(label="Retrieved sources", lines=14),
    ],
    title="Multimodal RAG Question Answering",
    description="Ask questions about the files indexed in your LanceDB vector store.",
    examples=[
        ["What are the steps to make Carbonara?"],
        ["Where should I travel in May or June?"],
        ["Which months are good for a trip to Japan's capital?"],
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3d147601c649cd59ed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Conclusion

🎉 **Congratulations!** You have successfully built an advanced **Multimodal RAG System** using LlamaIndex's `SimpleDirectoryReader` with comprehensive cross-modal capabilities.

## Student experiments (for trying out later after the session)

Complete these small experiments later to build intuition.

### Experiment 1: Change `similarity_top_k`

Try values like 2, 5, and 10.

Question to answer:

- Does increasing `top_k` always improve the answer?
- Do you see more noise when `top_k` is high?

### Experiment 2: Ask the same question in different wording

Example:

- "What is the refund policy?"
- "Can customers get their money back?"

Question to answer:

- Does semantic search still retrieve similar chunks?

### Experiment 3: Inspect sources before trusting the answer

Question to answer:

- Did the LLM answer using the right sources?
- Were any irrelevant chunks included?

In [ ]:
# Experiment area
# Change the query and top_k values below.

experiment_query = "Replace this with your own question"
experiment_top_k = 3

experiment_retriever = create_retriever(index, similarity_top_k=experiment_top_k)
experiment_nodes = retrieve_chunks(experiment_retriever, experiment_query)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why do we parse and load files before indexing?
2. Why do we chunk text instead of storing the full document as one unit?
3. What does `chunk_overlap` help with?
4. What is stored in the vector database?
5. What role does `StorageContext` play?
6. What is the difference between the retriever's output and the generated answer?
7. Give one example where returning retrieved chunks directly is better than generating an answer.
8. Give one example where generation is useful after retrieval.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.